# Spark Tutorials - Fabric Runtime 2.0

Fabric-native edition. The `spark` session and `SparkContext` are managed by Microsoft Fabric; do not create or stop them manually. Before running, select **Fabric Runtime 2.0** and attach a Lakehouse that you can write to. Any workspace and Lakehouse are supported; this checked-in notebook contains no tenant-specific IDs. The first code cell verifies Apache Spark 4.1 and prints the active Fabric context.


# Spark Tutorials — runnable PySpark 4.1 course

This notebook follows every entry in the official [PySpark 4.1 Tutorials](https://spark.apache.org/docs/4.1.0/api/python/tutorial/index.html).
Directly supported examples execute and assert exact results. Cluster deployment,
optional Python-worker, external database, and pandas-on-Spark workflows use bounded
capability probes plus copyable recipes rather than mutating this shared playground.

Coverage: package management; Spark SQL; Arrow conversion; Arrow UDFs; vectorized
and row-oriented UDTFs; Python Data Source API; Python/Spark type conversion; all
pandas-on-Spark tutorial pages; and handoffs to SQL, streaming, and ML guides.


## 0. Start a remote session and build bounded fixtures

Microsoft Fabric Spark keeps plans remote. Every collection below is deliberately tiny.


In [ ]:
# Microsoft Fabric injects the Spark session.
import importlib.util
import os
import sys

import pyspark
from notebookutils import runtime
from pyspark.sql import Row, Window, functions as F, types as T

spark = globals()["spark"]
fabric_context = runtime.context
actual_environment_id = fabric_context.get("environmentId")
assert spark.version.startswith("4.1"), (
    f"Fabric Runtime 2.0 requires Apache Spark 4.1; got {spark.version}"
)
print({
    "fabric_workspace_id": fabric_context.get("currentWorkspaceId"),
    "fabric_notebook": fabric_context.get("currentNotebookName"),
    "environment_id": actual_environment_id,
    "spark": spark.version,
    "application_id": spark.sparkContext.applicationId,
})

import os
import sys

from pyspark.sql import SparkSession, functions as F, types as T
sales = spark.createDataFrame([
    (1, "west", "book", 2, 12.50),
    (2, "east", "pen", 5, 2.00),
    (3, "west", "pen", 3, 2.00),
    (4, "north", "book", 1, 12.50),
], "id LONG, region STRING, product STRING, quantity INT, price DOUBLE")
sales.createOrReplaceTempView("tutorial_sales")
assert sales.count() == 4
print({"client": pyspark.__version__, "server": spark.version, "fabric_runtime": "2.0"})


## 1. [Python Package Management](https://spark.apache.org/docs/4.1.0/api/python/tutorial/python_packaging.html)

Driver and every executor need compatible Python packages. The official methods are
[Using PySpark Native Features](https://spark.apache.org/docs/4.1.0/api/python/tutorial/python_packaging.html#using-pyspark-native-features),
[Using Conda](https://spark.apache.org/docs/4.1.0/api/python/tutorial/python_packaging.html#using-conda),
[Using Virtualenv](https://spark.apache.org/docs/4.1.0/api/python/tutorial/python_packaging.html#using-virtualenv),
[Using PEX](https://spark.apache.org/docs/4.1.0/api/python/tutorial/python_packaging.html#using-pex), and
[Using uv run](https://spark.apache.org/docs/4.1.0/api/python/tutorial/python_packaging.html#using-uv-run).

Native `--py-files` ships `.py`, `.zip`, and `.egg` code but not native wheels.
Conda/venv archives ship an interpreter environment; PEX ships a self-contained
Python executable; `uv run` resolves a locked project. Never set
`PYSPARK_DRIVER_PYTHON` in YARN/Kubernetes cluster mode. Version-lock Python,
PySpark, pandas, and PyArrow together.


In [ ]:
packaging_recipes = {
    "native": "spark-submit --py-files app.zip app.py",
    "conda": "spark-submit --archives env.tar.gz#environment app.py",
    "virtualenv": "spark-submit --archives venv.tar.gz#environment app.py",
    "pex": "PYSPARK_PYTHON=./app.pex spark-submit --files app.pex app.py",
    "uv": "uv run --frozen spark-submit app.py",
}
assert set(packaging_recipes) == {"native", "conda", "virtualenv", "pex", "uv"}
print({"python": sys.version.split()[0], "recipes": packaging_recipes})


## 2. [Spark SQL](https://spark.apache.org/docs/4.1.0/api/python/tutorial/sql/index.html): query, DataFrame, and temporary-view interoperability

SQL and DataFrame expressions share one optimizer. Use SQL for declarative relational
logic and DataFrame APIs for programmatic composition; compare results, not syntax.


In [ ]:
sql_totals = spark.sql("""SELECT region, SUM(quantity * price) AS revenue
  FROM tutorial_sales
  GROUP BY region""")
api_totals = sales.groupBy("region").agg(
    F.sum(F.col("quantity") * F.col("price")).alias("revenue")
)
expected = [("east", 10.0), ("north", 12.5), ("west", 31.0)]
assert [tuple(row) for row in sql_totals.orderBy("region").collect()] == expected
assert sql_totals.orderBy("region").collect() == api_totals.orderBy("region").collect()
sql_totals.orderBy("region").show()


## 3. [Apache Arrow in PySpark](https://spark.apache.org/docs/4.1.0/api/python/tutorial/sql/arrow_pandas.html)

Arrow transfers columnar data between Python and Spark. `toArrow` and `toPandas`
collect all rows, so aggregate or limit first. The relevant controls are
`spark.sql.execution.arrow.pyspark.enabled` and
`spark.sql.execution.arrow.pyspark.fallback.enabled`.


In [ ]:
import pyarrow as pa

arrow_input = pa.table({"id": [1, 2], "value": ["a", "b"]})
arrow_df = spark.createDataFrame(arrow_input)
arrow_output = arrow_df.orderBy("id").toArrow()
assert arrow_output.to_pydict() == {"id": [1, 2], "value": ["a", "b"]}
print({"rows": arrow_output.num_rows, "schema": str(arrow_output.schema)})


## 4. [Arrow Python UDFs](https://spark.apache.org/docs/4.1.0/api/python/tutorial/sql/arrow_python_udf.html)

Arrow UDFs operate on `pyarrow.Array` values; scalar Arrow optimization uses
`@udf(..., useArrow=True)`. This client exposes both APIs, but this lightweight
server lacks its Python-worker bridge. The probe verifies execution, not `hasattr`.


In [ ]:
@F.udf(returnType="long", useArrow=True)
def arrow_plus_one(value):
    return None if value is None else value + 1

try:
    values = [row.value for row in spark.range(2).select(
        arrow_plus_one("id").alias("value")
    ).collect()]
    arrow_udf_capability = {"supported": True, "values": values}
except Exception as error:
    arrow_udf_capability = {"supported": False, "reason": str(error).
      splitlines()[0][:220]}
print(arrow_udf_capability)


## 5. [Vectorized Python UDTFs](https://spark.apache.org/docs/4.1.0/api/python/tutorial/sql/arrow_python_udtf.html)

Vectorized UDTFs exchange Arrow batches and can yield multiple output batches.
They require Python-worker and Arrow support on the server. Native generators remain
optimizer-visible alternatives for arrays and maps.


In [ ]:
native_rows = [tuple(row) for row in spark.tvf.explode(
    F.array(F.lit(10), F.lit(20))
).collect()]
assert native_rows == [(10,), (20,)]
print({"native_generator": native_rows, "arrow_udtf_module": hasattr(F, "udtf")})


## 6. [Python UDTFs](https://spark.apache.org/docs/4.1.0/api/python/tutorial/sql/python_udtf.html)

A Python UDTF class implements `eval`, declares a return schema, and is invoked in
a SQL `FROM` clause. Registration is session-scoped. This probe records whether the
Fabric Spark session can resolve a client-registered function.


In [ ]:
@F.udtf(returnType="number: INT")
class CountDown:
    def eval(self, start: int):
        while start > 0:
            yield (start,)
            start -= 1

try:
    spark.udtf.register("tutorial_count_down", CountDown)
    values = [row.number for row in spark.sql(
        "SELECT * FROM tutorial_count_down(3)"
    ).collect()]
    udtf_capability = {"supported": True, "values": values}
except Exception as error:
    udtf_capability = {"supported": False, "reason": str(error).splitlines()[0][:220]}
print(udtf_capability)


## 7. [Python Data Source API](https://spark.apache.org/docs/4.1.0/api/python/tutorial/sql/python_data_source.html)

Python data sources implement batch/stream reader and writer interfaces, partitions,
and commit messages, then register through `spark.dataSource`. A real source needs
external storage and worker execution, so this cell inventories the installed types.


In [ ]:
import pyspark.sql.datasource as datasource

data_source_types = {
    name: hasattr(datasource, name)
    for name in (
        "DataSource", "DataSourceReader", "DataSourceStreamReader",
        "DataSourceWriter", "InputPartition", "WriterCommitMessage",
    )
}
assert data_source_types["DataSource"] and hasattr(spark, "dataSource")
print(data_source_types)


## 8. [Python to Spark Type Conversions](https://spark.apache.org/docs/4.1.0/api/python/tutorial/sql/type_conversions.html)

Explicit schemas control integer widths, decimal precision, binary values, temporal
types, arrays, maps, structs, and nullability. Important controls include nested-dict
inference, timestamp type, pandas dict inference, and binary-as-bytes behavior.


In [ ]:
import datetime
import decimal

conversion_schema = T.StructType([
    T.StructField("small", T.ByteType(), False),
    T.StructField("amount", T.DecimalType(8, 2), False),
    T.StructField("day", T.DateType(), False),
    T.StructField("payload", T.BinaryType(), False),
    T.StructField("labels", T.ArrayType(T.StringType()), False),
    T.StructField("attrs", T.MapType(T.StringType(), T.LongType()), False),
])
converted = spark.createDataFrame([(
    7, decimal.Decimal("12.34"), datetime.date(2025, 1, 2),
    b"spark", ["a", "b"], {"x": 1},
)], conversion_schema)
row = converted.first()
assert row.small == 7 and row.amount == decimal.Decimal("12.34")
assert row.payload == bytearray(b"spark") or row.payload == b"spark"
print(converted.schema.simpleString(), row.asDict())


## 9. [Pandas API on Spark](https://spark.apache.org/docs/4.1.0/api/python/tutorial/pandas_on_spark/index.html): capability boundary

Pandas API on Spark distributes pandas-like operations through Spark. It requires a
compatible pandas/PyArrow matrix. This environment exposes the module but the installed
pandas version is newer than its adapter; all following pandas sections therefore pair
canonical patterns with executable Spark DataFrame equivalents.


In [ ]:
pandas_on_spark_installed = importlib.util.find_spec("pyspark.pandas") is not None
try:
    import pyspark.pandas as ps
    pandas_on_spark_probe = {"importable": True, "sum": int(ps.Series([1, 2]).sum())}
except Exception as error:
    pandas_on_spark_probe = {
        "installed": pandas_on_spark_installed,
        "importable": False,
        "reason": str(error).splitlines()[0][:220],
    }
print(pandas_on_spark_probe)


## 10. [Options and settings](https://spark.apache.org/docs/4.1.0/api/python/tutorial/pandas_on_spark/options.html)

Use `ps.get_option`, `set_option`, `reset_option`, and `option_context` for pandas-on-
Spark behavior. Spark SQL options still control execution. Never set global options in
reusable libraries without restoring them.


In [ ]:
option_contracts = {
    "compute.default_index_type": "distributed-sequence",
    "compute.ops_on_diff_frames": "explicitly opt in when needed",
    "display.max_rows": "display only, not an execution limit",
    "spark.sql.ansi.enabled": "type coercion and failure semantics",
}
assert "compute.default_index_type" in option_contracts
print(option_contracts)


## 11. [From/to pandas and PySpark DataFrames](https://spark.apache.org/docs/4.1.0/api/python/tutorial/pandas_on_spark/pandas_pyspark.html)

Canonical conversions are `ps.from_pandas`, `psdf.to_pandas`, `psdf.to_spark`, and
`DataFrame.pandas_api`. Conversion to local pandas collects data. The bounded Arrow
conversion below provides the same safety lesson without importing the incompatible adapter.


In [ ]:
bounded_arrow = sales.orderBy("id").limit(4).toArrow()
assert bounded_arrow.num_rows == 4
print({"arrow_rows": bounded_arrow.num_rows, "columns": bounded_arrow.column_names})


## 12. [Transform and apply a function](https://spark.apache.org/docs/4.1.0/api/python/tutorial/pandas_on_spark/transform_apply.html)

pandas-on-Spark distinguishes `transform` (same-length output), `apply`,
`transform_batch`, and `apply_batch`; type hints avoid expensive schema inference.
Prefer native Spark expressions when possible.


In [ ]:
transformed = sales.select(
    "id", F.upper("product").alias("product"),
    (F.col("quantity") * F.col("price")).alias("revenue"),
).orderBy("id")
values = [tuple(row) for row in transformed.collect()]
assert values[0] == (1, "BOOK", 25.0) and len(values) == 4
transformed.show()


## 13. [Type Support](https://spark.apache.org/docs/4.1.0/api/python/tutorial/pandas_on_spark/types.html) and [Type Hints](https://spark.apache.org/docs/4.1.0/api/python/tutorial/pandas_on_spark/typehints.html)

pandas dtypes map to Spark SQL types with caveats for nullable integers, categorical,
timestamps, decimals, arrays, structs, and indexes. Function annotations can declare
Series/DataFrame output shape and prevent exploratory execution for schema inference.


In [ ]:
type_support = {
    "int64": "LongType", "float64": "DoubleType", "bool": "BooleanType",
    "datetime64[ns]": "TimestampType", "object/string": "StringType or inferred",
    "annotations": "declare return dtype/schema to avoid inference jobs",
}
assert type_support["int64"] == "LongType"
print(type_support)


## 14. [From/to other DBMSes](https://spark.apache.org/docs/4.1.0/api/python/tutorial/pandas_on_spark/from_to_dbms.html)

The tutorial uses pandas database readers/writers for small local transfers. Distributed
Spark workloads should use JDBC readers/writers with predicates or partition bounds,
fetch size, transaction semantics, and secret management. No external database is mutated.


In [ ]:
jdbc_contract = {
    "reader": all(hasattr(spark.read, name) for name in ("jdbc", "format")),
    "writer": hasattr(sales.write, "jdbc"),
    "required": ["url", "dbtable/query", "credentials", "partition strategy"],
    "executed": False,
}
assert jdbc_contract["reader"] and jdbc_contract["writer"]
print(jdbc_contract)


## 15. [Best Practices](https://spark.apache.org/docs/4.1.0/api/python/tutorial/pandas_on_spark/best_practices.html)

Inspect plans; avoid unnecessary shuffles and single-partition work; checkpoint overly
complex plans; avoid reserved `__...__` columns; use distributed/default indexes wisely;
and avoid collecting large results. These principles apply directly to DataFrames.


In [ ]:
best_practice_plan = sales.filter("quantity > 1").groupBy("region").agg(
    F.sum(F.col("quantity") * F.col("price")).alias("revenue")
)
best_practice_plan.explain(mode="simple")
assert best_practice_plan.count() == 2


## 16. [Supported pandas API](https://spark.apache.org/docs/4.1.0/api/python/tutorial/pandas_on_spark/supported_pandas_api.html)

Support is version-specific. Check the official matrix before porting code, especially
index, categorical, plotting, resampling, rolling, missing-value, and extension APIs.
Pin versions and test semantic differences rather than relying on attribute presence.


In [ ]:
supported_api_checklist = [
    "DataFrame/Series", "Index/MultiIndex", "GroupBy", "Window/Rolling",
    "Resample", "I/O", "Plotting", "Missing data", "General functions",
]
assert len(supported_api_checklist) == 9
print(supported_api_checklist)


## 17. [FAQ](https://spark.apache.org/docs/4.1.0/api/python/tutorial/pandas_on_spark/faq.html)

Common issues include expensive default indexes, operations across different frames,
plotting/collection limits, missing APIs, SQL configuration, Arrow dependencies, and
slow plans. Diagnose with `spark.explain`, bounded samples, and versioned tests.


In [ ]:
faq_diagnostics = {
    "client": pyspark.__version__, "server": spark.version,
    "fabric_runtime": "2.0", "pandas_probe": pandas_on_spark_probe,
    "next_notebooks": ["22_performance_tuning_troubleshooting.ipynb",
      "23_query_xray.ipynb"],
}
assert faq_diagnostics["fabric_runtime"] == "2.0"
print(faq_diagnostics)


## 18. Broader Spark programming-guide handoffs

The tutorial index also links the Spark SQL/DataFrames/Datasets guide, Structured
Streaming guide, and MLlib guide. Continue with notebooks **32_spark_sql_reference**,
**18_structured_streaming**, and **17_machine_learning** for bounded runnable courses.


In [ ]:
handoffs = {
    "sql": "32_spark_sql_reference.ipynb",
    "streaming": "18_structured_streaming.ipynb",
    "machine_learning": "17_machine_learning.ipynb",
}
assert all(name.endswith(".ipynb") for name in handoffs.values())
print(handoffs)


## 19. Capstone: combine SQL, types, nested data, Arrow, and cleanup


In [ ]:
report = spark.sql("""SELECT region, product, SUM(quantity) AS units,
         CAST(SUM(quantity * price) AS DECIMAL(12, 2)) AS revenue
  FROM tutorial_sales
  GROUP BY region, product""")
report = report.withColumn("summary", F.struct("units", "revenue"))
report = report.orderBy("region", "product")
rows = report.collect()
arrow_report = report.toArrow()
assert len(rows) == 4 and arrow_report.num_rows == 4
assert sum(row.units for row in rows) == 11
for row in rows:
    print(row.asDict())
spark.catalog.dropTempView("tutorial_sales")
# Fabric owns the Spark session; do not call spark.stop().


## Tutorial checklist

- Package Python dependencies for every executor and pin one environment.
- Prefer SQL/DataFrame built-ins; test optional Python execution features.
- Bound Arrow/pandas collection and declare schemas/types explicitly.
- Treat indexes, shuffles, options, and DBMS writes as distributed contracts.
- Inspect plans, test version compatibility, and use specialized notebooks for
  streaming, ML, lakehouse, and deep SQL workflows.
